# 03 · Retrieval-augmented generation (RAG)

Pipeline: **chunk → embed → index → retrieve → build a prompt with the context → answer with citations**.

Evaluate retrieval and answers separately. A wrong answer can come from retrieval
(the right document was never retrieved) or from generation (the document was retrieved
but the answer misused it).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from rag import RAGPipeline, chunk_text
from embeddings import get_embedder
from llm_utils import get_llm
from sample_data import DOCS, GOLD_RETRIEVAL
from evaluation import retrieval_metrics, doc_ids_from_chunks

print("Documents:", list(DOCS))

## 1. Chunking

In [ ]:
sample = DOCS["refund_policy"]
for i, chunk in enumerate(chunk_text(sample, max_words=12, overlap=4)):
    print(i, "|", chunk)

## 2. Index the documents

In [ ]:
pipeline = RAGPipeline(get_llm(), get_embedder(), k=2, max_words=40, overlap=10)
print("Chunks indexed:", pipeline.index(DOCS))

## 3. Retrieval quality on a small gold set

In [ ]:
for question, relevant_doc in GOLD_RETRIEVAL:
    hits = pipeline.retrieve(question, k=4)
    ranked = doc_ids_from_chunks([h["id"] for h in hits])
    print(f"{question}\n  ranked docs: {ranked}")
    print(f"  {retrieval_metrics(ranked, [relevant_doc], k=2)}\n")

## 4. Answers with citations

In [ ]:
for question in [
    "How long do refunds take?",
    "What does the warranty cover?",
    "Do you sell furniture?",
]:
    result = pipeline.answer(question)
    print("Q:", question)
    print("A:", result["answer"])
    print("Sources:", result["sources"], "\n")

The last question is not covered by the documents, so the system should say it does not know.
Test this behaviour across many out-of-scope questions before you trust it, because
models often answer from general knowledge when the context is thin.

## 5. Inspect exactly what the model received

In [ ]:
result = pipeline.answer("When can I call support by phone?")
for m in result["messages"]:
    print(f"--- {m['role']} ---\n{m['content']}\n")

## Tuning levers

- **Chunk size and overlap:** smaller chunks give precise matches; larger chunks keep context. Overlap protects facts that straddle a boundary.
- **k:** more chunks raise recall but add noise and cost.
- **Embedding model:** semantic models usually beat lexical ones for paraphrased questions.
- **Metadata filters:** restrict by product, date, or access level before ranking.
- **Reranking and hybrid search:** combine keyword and vector scores, then rerank the top results with a stronger model.
- **Prompt instructions:** require citations and abstention, and test both.

Change one lever at a time and re-run the evaluation in notebook 05.